# 🎤 KaraokeAI Backend Server
### Run this notebook to start the backend API for your KaraokeAI app
**Steps:** Click **Runtime → Run all** after setting your ngrok token in Cell 4.
- After Cell 5 runs, you will get a **public URL** — copy and paste it into your app!

In [ ]:
# ── CELL 1: Install all dependencies ──────────────────────────
print('Installing dependencies... (takes 2-3 minutes)')
!pip install -q faster-whisper demucs ffmpeg-python pysubs2 fastapi uvicorn pyngrok nest-asyncio transliterate anyascii aksharamukha
!pip install -q 'fastapi[standard]'
!apt-get install -q ffmpeg
print('✅ All dependencies installed!')

In [ ]:
# ── CELL 2: Clone your GitHub repository ──────────────────────
import os

GITHUB_REPO = 'https://github.com/shanmuganathan457/karaoke-video-generator.git'

if os.path.exists('karaoke-video-generator'):
    print('Repo already exists, pulling latest changes...')
    os.system('cd karaoke-video-generator && git pull')
else:
    print('Cloning your GitHub repository...')
    os.system(f'git clone {GITHUB_REPO}')

os.chdir('karaoke-video-generator')
print(f'✅ Working directory: {os.getcwd()}')
print('📁 Files found:', os.listdir('.'))

In [ ]:
# ── CELL 3: Write the FastAPI server ──────────────────────────
import os

lines = [
    'import sys, os, uuid\n',
    'sys.path.insert(0, os.getcwd())\n',
    '\n',
    '# Monkey Patch: fix av library bug on Python 3.13\n',
    'import av\n',
    '_orig_av_open = av.open\n',
    'def _patched_av_open(*args, **kwargs):\n',
    '    kwargs.pop("metadata_errors", None)\n',
    '    return _orig_av_open(*args, **kwargs)\n',
    'av.open = _patched_av_open\n',
    '\n',
    'from fastapi import FastAPI, UploadFile, File, Form\n',
    'from fastapi.responses import FileResponse, JSONResponse\n',
    'from fastapi.middleware.cors import CORSMiddleware\n',
    'from fastapi.staticfiles import StaticFiles\n',
    'from pydantic import BaseModel\n',
    'from typing import List, Dict, Any\n',
    '\n',
    'from karaoke_generator.transcriber import AudioTranscriber\n',
    'from karaoke_generator.subtitle_generator import SubtitleGenerator\n',
    'from karaoke_generator.video_processor import VideoProcessor\n',
    'from karaoke_generator.vocal_separator import VocalSeparator\n',
    '\n',
    'app = FastAPI(title="KaraokeAI Backend API")\n',
    'app.add_middleware(\n',
    '    CORSMiddleware,\n',
    '    allow_origins=["*"],\n',
    '    allow_methods=["*"],\n',
    '    allow_headers=["*"],\n',
    ')\n',
    '\n',
    'os.makedirs("/tmp", exist_ok=True)\n',
    'app.mount("/files", StaticFiles(directory="/tmp"), name="files")\n',
    '\n',
    '@app.get("/")\n',
    'def root():\n',
    '    return {"status": "KaraokeAI Backend is running!", "version": "2.0.0"}\n',
    '\n',
    '@app.post("/generate")\n',
    'async def generate_karaoke(\n',
    '    video: UploadFile = File(...),\n',
    '    language: str = Form(default="")\n',
    '):\n',
    '    job_id = str(uuid.uuid4())[:8]\n',
    '    input_path = f"/tmp/input_{job_id}.mp4"\n',
    '    temp_audio = f"/tmp/audio_{job_id}.wav"\n',
    '    vocals_audio = None\n',
    '    try:\n',
    '        with open(input_path, "wb") as f:\n',
    '            f.write(await video.read())\n',
    '        processor = VideoProcessor()\n',
    '        transcriber = AudioTranscriber()\n',
    '        processor.extract_audio(input_path, temp_audio)\n',
    '        vocals_audio, _ = VocalSeparator.separate(temp_audio)\n',
    '        result = transcriber.transcribe(\n',
    '            vocals_audio,\n',
    '            language=language if language else None,\n',
    '            vad_filter=False\n',
    '        )\n',
    '        return JSONResponse({\n',
    '            "job_id": job_id,\n',
    '            "video_url": f"/files/input_{job_id}.mp4",\n',
    '            "segments": result["segments"],\n',
    '            "language": result["language"]\n',
    '        })\n',
    '    except Exception as e:\n',
    '        return JSONResponse(status_code=500, content={"error": str(e)})\n',
    '\n',
    'class ExportRequest(BaseModel):\n',
    '    job_id: str\n',
    '    segments: List[Dict[Any, Any]]\n',
    '    font_size: int = 24\n',
    '    alignment: int = 2\n',
    '    romanize: bool = True\n',
    '    language: str = "en"\n',
    '\n',
    '@app.post("/export")\n',
    'async def export_video(req: ExportRequest):\n',
    '    input_path = f"/tmp/input_{req.job_id}.mp4"\n',
    '    output_ass = f"/tmp/karaoke_{req.job_id}.ass"\n',
    '    output_video = f"/tmp/output_{req.job_id}.mp4"\n',
    '    try:\n',
    '        sub_gen = SubtitleGenerator()\n',
    '        processor = VideoProcessor()\n',
    '        sub_gen.generate(req.segments, output_ass, romanize=req.romanize, language=req.language)\n',
    '        processor.burn_subtitles(input_path, output_ass, output_video)\n',
    '        return {"video_url": f"/files/output_{req.job_id}.mp4"}\n',
    '    except Exception as e:\n',
    '        return JSONResponse(status_code=500, content={"error": str(e)})\n'
]

with open('api_server.py', 'w') as f:
    f.writelines(lines)

print('✅ api_server.py written! Contents:')
print(open('api_server.py').read()[:300])

In [ ]:
# ── CELL 4: Enter your ngrok token ────────────────────────────
# Get your FREE token from: https://dashboard.ngrok.com/get-started/your-authtoken
# Paste your token below between the quotes:

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token set!')

In [ ]:
# ── CELL 5: TEST server imports before starting ────────────────
# This catches import errors BEFORE we run uvicorn in background
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-c', 'import api_server; print("✅ api_server imports OK")'],
    capture_output=True, text=True
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)
if result.returncode != 0:
    raise RuntimeError('api_server.py has import errors! Fix before starting.')

In [ ]:
# ── CELL 6: START THE SERVER ──────────────────────────────────
# This prints your public URL — copy it and paste into your React app!

import subprocess, time, os
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Start FastAPI server — logs go to /tmp/uvicorn.log so we can debug
log_file = open('/tmp/uvicorn.log', 'w')
proc = subprocess.Popen(
    ['uvicorn', 'api_server:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_file,
    stderr=log_file
)

# Wait for uvicorn to be ready
time.sleep(5)

# Check if it is still alive
if proc.poll() is not None:
    print('❌ uvicorn CRASHED! Error log:')
    print(open('/tmp/uvicorn.log').read())
    raise RuntimeError('uvicorn failed to start')

# Open a public tunnel on port 8000
public_url = ngrok.connect(8000)

print('=' * 60)
print('🚀 KaraokeAI Backend is LIVE!')
print('=' * 60)
print(f'\n📡 YOUR PUBLIC API URL:')
print(f'   {public_url}')
print(f'\n✅ Copy this URL and give it to your developer!')
print(f'\n⚠️  Keep this tab open — closing it stops the server.')
print('=' * 60)

# Keep cell alive
while True:
    time.sleep(60)
    # Print last 3 lines of uvicorn log
    try:
        log_lines = open('/tmp/uvicorn.log').readlines()
        print('💓 Server running |', ''.join(log_lines[-1:]).strip())
    except:
        print('💓 Server still running...')